# predicting_arma

> 对应代码：`EconometricsCode/code_in_notes/Chap.26/predicting_arma.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.26`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.26")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

我们设定样本量为 50，并固定随机种子以保证结果可复现。scalar sept=28 表示第 28 期是预测起点：28 期之前生成''历史''数据，28 期及以后展示预测路径。

In [ ]:
%%stata
clear
set obs 50
set seed 19880505
gen t=_n
tsset t
scalar sept=28

我们在历史区间（t<28）内生成冲击 e，并按 ARMA(1,1) 的数据生成过程 y_t = 3 + 0.5 y_{t-1} + e_t + 0.8 e_{t-1} 递归构造序列 y。注意 e 只在 t<28 时有值，因此 28 期以后 y 的''新息''为 0，这正是最优预测的做法：未来的冲击以其条件期望 0 代替。

In [ ]:
%%stata
gen e=rnormal() if t<sept
gen y=6 if t<sept
replace y=3+0.5*L.y+e+0.8*L.e if t<sept & t>1

在预测起点 t=28 处，由于 e_28 未知而被置为 0，但 e_27 已知，所以一步预测为 3 + 0.5 y_27 + 0.8 e_27；从 t>28 起，连滞后冲击也不再出现在方程中，多步预测退化为纯 AR(1) 递推 3 + 0.5 y_{t-1}。这体现了 MA 部分的影响只持续 q=1 期：预测越远，ARMA 的预测越接近其 AR 骨架。

In [ ]:
%%stata
replace y=3+0.5*L.y+0.8*L.e if t==sept
replace y=3+0.5*L.y if t>sept

接下来我们计算预测误差方差，用以构造预测区间。预测起点前一期（t=27）是已知数据，方差为 0；一步预测（t=28）的误差就是 e_28，方差为 1。

In [ ]:
%%stata
gen lb=.
gen ub=.
gen variance=.
replace variance=0 if t==sept-1
replace variance=1 if t==sept

对于 h 步预测（h>=2），ARMA(1,1) 的预测误差方差满足递推 Var = Var_{h-1} + (φ+θ)^2 φ^{2(h-2)}。这里 φ=0.5、θ=0.8，故 φ+θ=1.3、φ^2=0.25，我们用 forvalues 循环逐期累加这一项，得到每一期的预测误差方差。variance 随预测步长单调递增并收敛于无条件方差，反映了预测不确定性随期限累积。

In [ ]:
%%stata
local sept1=sept+1
forvalues i= `sept1'/50{
	replace variance=L.variance+1.3^2*0.25^(`i'-sept-1) if t==`i'
}

最后我们基于正态近似构造 95% 预测区间：上界为点预测加 1.96 倍标准差，下界为减 1.96 倍标准差。

In [ ]:
%%stata
replace ub=y+1.96*sqrt(variance) if t>=sept-1
replace lb=y-1.96*sqrt(variance) if t>=sept-1

画图：实线为历史数据，点划线为预测路径，虚线为预测区间的上下界，直观展示预测值回归均值、区间逐渐变宽的特征。

In [ ]:
%%stata
twoway (tsline y if t<=sept-1) (tsline y if t>=sept-1, lpattern(-.)) (tsline lb if t>=sept-1, lpattern(dash) lc(orange)) (tsline ub if t>=sept-1, lpattern(dash) lc(orange)), legend(off)